# ADVC — Colab Setup & Phase 1 Evaluation
Run cells **top to bottom** on a fresh Colab session.

> **Before starting:** Make sure GPU is enabled — Runtime → Change runtime type → T4 GPU

**Dataset:** ImageNette (10-class ImageNet subset, ~400 MB).  
Downloaded once to Google Drive — subsequent sessions just mount Drive and skip the download.

## 1 — Check GPU

In [ ]:
!nvidia-smi

## 2 — Clone repo & navigate

In [ ]:
%cd /content
!git clone https://github.com/Jmanav/ADVC.git ADVC-main
%cd ADVC-main

## 3 — Install dependencies

In [ ]:
!pip install -r requirements.txt

## 4 — Verify PyTorch + GPU

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('Torch version  :', torch.__version__)

## 5 — Mount Google Drive
ImageNette will be stored here permanently so you never download it twice.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 6 — Download ImageNette to Drive (skip if already done)
~400 MB download, runs once.  
If `/content/drive/MyDrive/ADVC/imagenette2-320/` already exists this cell does nothing.

In [ ]:
import os, pathlib

DRIVE_ADVC  = pathlib.Path('/content/drive/MyDrive/ADVC')
DRIVE_DATA  = DRIVE_ADVC / 'imagenette2-320'
TGZ_PATH    = '/content/imagenette2-320.tgz'

DRIVE_ADVC.mkdir(parents=True, exist_ok=True)

if DRIVE_DATA.exists():
    print('ImageNette already on Drive — skipping download.')
else:
    print('Downloading ImageNette (~400 MB) ...')
    !wget -q --show-progress \
        -O {TGZ_PATH} \
        https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
    print('Extracting to Drive ...')
    !tar -xzf {TGZ_PATH} -C {str(DRIVE_ADVC)}
    os.remove(TGZ_PATH)
    print('Done.')

!echo "Val classes: $(ls {str(DRIVE_DATA / 'val')} | wc -l)"
!echo "Val images : $(find {str(DRIVE_DATA / 'val')} -name '*.JPEG' | wc -l)"

## 7 — Link Drive data into the project
Creates `data/imagenet/val` → Drive path so the experiment scripts find the data.

In [ ]:
import pathlib, os

DRIVE_VAL = pathlib.Path('/content/drive/MyDrive/ADVC/imagenette2-320/val')
LOCAL_VAL = pathlib.Path('data/imagenet/val')

LOCAL_VAL.parent.mkdir(parents=True, exist_ok=True)

# Remove stale symlink or empty dir if present
if LOCAL_VAL.is_symlink() or LOCAL_VAL.exists():
    if LOCAL_VAL.is_symlink():
        LOCAL_VAL.unlink()
    # only remove if empty to avoid accidental data loss
    elif not any(LOCAL_VAL.iterdir()):
        LOCAL_VAL.rmdir()

if not LOCAL_VAL.exists():
    LOCAL_VAL.symlink_to(DRIVE_VAL)
    print(f'Linked {LOCAL_VAL} -> {DRIVE_VAL}')
else:
    print(f'{LOCAL_VAL} already set up.')

## 8 — Verify ImageFolder

In [ ]:
from torchvision.datasets import ImageFolder

ds = ImageFolder('data/imagenet/val')
print(f'Images  : {len(ds)}')
print(f'Classes : {len(ds.classes)}')
print(f'Synsets : {ds.classes}')

assert len(ds) >= 100, 'Too few images — check Drive link'
print('OK — ready to run experiments.')

## 9 — Run Phase 1 evaluation
Loops over `fp32 → int8 → int4` × `fgsm / pgd / patch`.  
Results are saved to `results/phase1_results.csv` after each combination.  
Safe to interrupt and re-run — completed rows are skipped automatically.

In [ ]:
!python experiments/eval_phase1.py --model deit_small

## 10 — View results

In [ ]:
import pandas as pd

df = pd.read_csv('results/phase1_results.csv')
df['clean_acc']       = df['clean_acc'].map('{:.4f}'.format)
df['robust_acc']      = df['robust_acc'].map('{:.4f}'.format)
df['asr']             = df['asr'].map('{:.4f}'.format)
df['robustness_gap']  = df['robustness_gap'].map('{:.4f}'.format)
df

## 11 — Save results to Google Drive
Colab wipes `/content` on runtime reset — copy results to Drive to persist them.

In [ ]:
import shutil, pathlib

dest = pathlib.Path('/content/drive/MyDrive/ADVC/results')
shutil.copytree('results', str(dest), dirs_exist_ok=True)
print(f'Results saved to {dest}')